## 1. Setup & Imports

Import libraries and load project configuration paths.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


In [2]:
from src.config import PROCESSED_DATA_DIR

FILES = [
    "cleaned_email_data_v1.csv",
    "cleaned_ceas_08.csv",
    "cleaned_spamassassin.csv",
    "cleaned_nazario.csv",
    "cleaned_nigerian_fraud.csv",
]


## 2. Load Cleaned Datasets

Load all individual cleaned CSV files, add a `source` column to track origin, and concatenate into one DataFrame.

In [3]:
frames = []
for f in FILES:
    path = PROCESSED_DATA_DIR / f
    df = pd.read_csv(path)
    print(f"{f:40s} shape={str(df.shape):12s}  0={df['label'].value_counts().get(0, 0):>6d}  1={df['label'].value_counts().get(1, 0):>6d}")
    frames.append(df)


cleaned_email_data_v1.csv                shape=(78260, 11)   0= 39245  1= 39015
cleaned_ceas_08.csv                      shape=(33066, 11)   0= 17134  1= 15932
cleaned_spamassassin.csv                 shape=(5328, 11)    0=  3638  1=  1690
cleaned_nazario.csv                      shape=(1534, 11)    0=     0  1=  1534
cleaned_nigerian_fraud.csv               shape=(3249, 11)    0=     0  1=  3249


In [4]:
db = pd.concat(frames, ignore_index=True)
print(f"Combined shape (before dedup): {db.shape}")
print(f"Label distribution:\n{db['label'].value_counts()}")


Combined shape (before dedup): (121437, 11)
Label distribution:
label
1    61420
0    60017
Name: count, dtype: int64


## 3. Deduplication

Remove duplicate email texts across all combined datasets.

In [5]:
db.drop_duplicates(subset="text", inplace=True)
print(f"Shape after dedup: {db.shape}")
print(f"Label distribution:\n{db['label'].value_counts()}")


Shape after dedup: (121084, 11)
Label distribution:
label
1    61083
0    60001
Name: count, dtype: int64


In [6]:
db.info()


<class 'pandas.DataFrame'>
Index: 121084 entries, 0 to 121436
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   label            121084 non-null  int64  
 1   text             121084 non-null  str    
 2   num_urls         121084 non-null  int64  
 3   num_exclamation  121084 non-null  int64  
 4   num_question     121084 non-null  int64  
 5   num_dollar       121084 non-null  int64  
 6   num_all_caps     121084 non-null  int64  
 7   num_numbers      121084 non-null  int64  
 8   word_count       121084 non-null  int64  
 9   capital_ratio    121084 non-null  float64
 10  emoji_count      121084 non-null  int64  
dtypes: float64(1), int64(9), str(1)
memory usage: 11.1 MB


## 4. Save Combined Dataset

Write the final combined and deduplicated dataset to the processed data directory.

In [7]:
out = PROCESSED_DATA_DIR / "cleaned_combined.csv"
db.to_csv(out, index=False)
print(f"Saved to {out}")


Saved to C:\Users\aungm\university\computing\cos30049-email-spam-detection\data\processed\cleaned_combined.csv
